In [82]:
import pandas as pd
from xml.dom import minidom
import re
from transformers import AutoTokenizer, RobertaModel, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import torch

# Reading the Data

In [93]:
platinum_df = pd.read_table('./data/matres/platinum.txt', header=None, sep='\t', names=['docid', 'verb1', 'verb2', 'eiid1', 'eiid2', 'relation'])
platinum_df[['eiid1', 'eiid2']] = 'E' + platinum_df[['eiid1', 'eiid2']].astype(str)

# Fixing a typo in the docid
platinum_df.loc[platinum_df['docid'] == 'nyt_20130321_sarcozy', 'docid'] = 'nyt_20130321_sarkozy'

platinum_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE


In [94]:
# Augment BEFORE/AFTER rows by adding swapped inverse examples
mask = platinum_df["relation"].isin(["BEFORE", "AFTER", "VAGUE", "EQUAL"])
swapped_df = platinum_df.loc[mask].copy()

# Swap event pair columns
swapped_df[["verb1", "verb2"]] = swapped_df[["verb2", "verb1"]].to_numpy()
swapped_df[["eiid1", "eiid2"]] = swapped_df[["eiid2", "eiid1"]].to_numpy()

# Invert relation
swapped_df["relation"] = swapped_df["relation"].map({"BEFORE": "AFTER", "AFTER": "BEFORE", "VAGUE": "VAGUE", "EQUAL": "EQUAL"})

platinum_df = pd.concat([platinum_df, swapped_df], ignore_index=True)

print(f"Added {len(swapped_df)} swapped rows")
print(f"New total rows: {len(platinum_df)}")

Added 837 swapped rows
New total rows: 1674


In [95]:
def purify_text(text):
    # Strip TIMEX3 attributes
    text = re.sub(r'<TIMEX3\b[^>]*/>', '[TIMEX3/]', text)
    text = re.sub(r'<TIMEX3\b[^>]*>', '[TIMEX3]', text)
    text = re.sub(r'</TIMEX3>', '[/TIMEX3]', text)

    event_stack = []

    # Replace EVENT tags with their eid values, capitalized
    def _replace_event_tag(match):
        tag = match.group(0)

        if tag.startswith('</EVENT'):
            if event_stack:
                return f"[/{event_stack.pop()}]"
            return tag

        eid_match = re.search(r'\beid\s*=\s*"([^"]+)"', tag)
        if eid_match:
            eid = eid_match.group(1)
            eid = eid.upper()
            event_stack.append(eid)
            return f"[{eid}]"
        return tag

    text = re.sub(r'</EVENT>|<EVENT\b[^>]*>', _replace_event_tag, text)

    return text

def split_sentences(text):
    sentences = re.split(r'(?<!\bMr\.)(?<!\bMs\.)(?<!\bMrs\.)(?<=[.!?])\s+', text)
    return sentences

In [96]:
unique_docids = platinum_df['docid'].unique()

doc_texts = {}
for docid in unique_docids:
    filename = f"{docid}.tml"
    text_content = minidom.parse('./data/tempeval/' + filename).getElementsByTagName('TEXT')[0]
    text_content_str = ''.join(node.toxml() for node in text_content.childNodes).strip()
    text_content_str = purify_text(text_content_str)
    sentences = split_sentences(text_content_str)
    doc_texts[docid] = {
        'text': text_content_str,
        'sentences': sentences
    }

docs_df = pd.DataFrame.from_dict(doc_texts, orient='index').reset_index()
docs_df.columns = ['docid', 'text', 'sentences']
docs_df = docs_df.set_index('docid')
docs_df.head()

,text,sentences
docid,,
WSJ_20130322_159,Israeli Prime Minister Benjamin Netanyahu [E1]...,[Israeli Prime Minister Benjamin Netanyahu [E1...
nyt_20130322_strange_computer,Our [TIMEX3]digital[/TIMEX3] age is all about ...,[Our [TIMEX3]digital[/TIMEX3] age is all about...
CNN_20130321_821,Barack Obama would [E1]make[/E1] a great stand...,[Barack Obama would [E1]make[/E1] a great stan...
nyt_20130321_cyprus,A Cyprus [E2001]exit[/E2001] from the euro uni...,[A Cyprus [E2001]exit[/E2001] from the euro un...
bbc_20130322_1353,Israel's prime minister has [E1]apologised[/E1...,[Israel's prime minister has [E1]apologised[/E...


# Creating Context

In [97]:
def create_context_window(sentences, eiid1, eiid2, padding = 1):
    sentence_indices = []
    for i, sentence in enumerate(sentences):
        if f'[{eiid1}]' in sentence or f'[{eiid2}]' in sentence:
            sentence_indices.append(i)

    if not sentence_indices:
        return ""

    start_index = max(0, min(sentence_indices) - padding)
    end_index = min(len(sentences), max(sentence_indices) + padding + 1)

    context_window = ' '.join(sentences[start_index:end_index])

    context_window = re.sub(rf'\[{re.escape(eiid1)}\]', '[T1]', context_window)
    context_window = re.sub(rf'\[/{re.escape(eiid1)}\]', '[/T1]', context_window)
    context_window = re.sub(rf'\[{re.escape(eiid2)}\]', '[T2]', context_window)
    context_window = re.sub(rf'\[/{re.escape(eiid2)}\]', '[/T2]', context_window)

    # Remove all other event tags like [E3], [/E3], etc.
    context_window = re.sub(r'\[/?E\d+\]', '', context_window)

    # Clean extra whitespace
    context_window = re.sub(r'\s+', ' ', context_window).strip()
    return context_window

In [98]:
platinum_df['context_window'] = platinum_df.apply(lambda row: create_context_window(docs_df.loc[row['docid'], 'sentences'], row['eiid1'], row['eiid2']), axis=1)
platinum_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation,context_window
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE,Israeli Prime Minister Benjamin Netanyahu [T1]...
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE,Israeli Prime Minister Benjamin Netanyahu [T1]...
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE,Israeli Prime Minister Benjamin Netanyahu [T1]...
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE,Israeli Prime Minister Benjamin Netanyahu [T1]...
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE,Israeli Prime Minister Benjamin Netanyahu apol...


In [99]:
# Stable mapping: relation label -> integer id
relation2id = {"VAGUE": 0, "BEFORE": 1, "AFTER": 2, "EQUAL": 3}
id2relation = {i: rel for rel, i in relation2id.items()}

num_labels = len(relation2id)

platinum_df["relation_id"] = platinum_df["relation"].map(relation2id)

In [100]:
# 90% train, 5% validation, 5% test (stratified by label)
train_df, temp_df = train_test_split(
    platinum_df,
    test_size=0.1,
    random_state=42,
    stratify=platinum_df["relation_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["relation_id"]
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 1506
Validation: 84
Test: 84


# Assemble the Model

In [101]:
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")
special = ["[T1]", "[/T1]", "[T2]", "[/T2]", "[TIMEX3]", "[/TIMEX3]"]
tokenizer.add_special_tokens({"additional_special_tokens": special})

6

In [102]:
class TemporalRelationsModel(torch.nn.Module):
    def __init__(self, num_labels, tokenizer):
        super(TemporalRelationsModel, self).__init__()
        
        config = AutoConfig.from_pretrained("FacebookAI/roberta-base")
        config.is_decoder = False
        self.roberta = RobertaModel(config=config)
        self.roberta.resize_token_embeddings(len(tokenizer))
        self.dropout = torch.nn.Dropout(self.roberta.config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(self.roberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(outputs.last_hidden_state[:, 0, :])
        logits = self.classifier(cls_output)
        return logits

In [103]:
# Tokenize context windows from the training split
train_encodings = tokenizer(
    train_df["context_window"].tolist(),
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

train_input_ids = train_encodings["input_ids"]
train_attention_mask = train_encodings["attention_mask"]

train_relation_ids = torch.tensor(train_df["relation_id"].tolist())

# Create training sequence (dataset + loader)
train_dataset = torch.utils.data.TensorDataset(
    train_input_ids,
    train_attention_mask,
    train_relation_ids
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)

print("input_ids shape:", train_input_ids.shape)
print("relation_ids shape:", train_relation_ids.shape)
print("num training batches:", len(train_loader))

input_ids shape: torch.Size([1506, 223])
relation_ids shape: torch.Size([1506])
num training batches: 95


In [104]:
t1_id = tokenizer.convert_tokens_to_ids("[T1]")
t2_id = tokenizer.convert_tokens_to_ids("[T2]")

ids = train_encodings["input_ids"][0].tolist()
assert t1_id in ids and t2_id in ids, "Truncated away a target event!"

In [105]:
val_encodings = tokenizer(
    val_df["context_window"].tolist(),
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)
val_labels = torch.tensor(val_df["relation_id"].values, dtype=torch.long)
val_dataset = torch.utils.data.TensorDataset(
    val_encodings["input_ids"],
    val_encodings["attention_mask"],
    val_labels
)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False)

In [106]:
loss_fn = torch.nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for input_ids, attention_mask, train_relation_ids in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = train_relation_ids.to(device)  # (B,) long

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)  # (B, C)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)

        preds = torch.argmax(logits, dim=-1)

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return {
        "val_loss": total_loss / len(loader.dataset),
        "macro_f1": macro_f1,
        "accuracy": acc,
    }

In [107]:
# Build model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TemporalRelationsModel(num_labels=num_labels, tokenizer=tokenizer).to(device)

# Cross-entropy setup
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Train
epochs = 6
for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)

    val_metrics = evaluate(model, val_loader, device)
    val_loss = val_metrics["val_loss"]
    val_f1 = val_metrics["macro_f1"]
    val_acc = val_metrics["accuracy"]

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/6 | Train Loss: 1.2058 | Val Loss: 1.2056 | Val F1: 0.1441 | Val Acc: 0.4048
Epoch 2/6 | Train Loss: 1.1732 | Val Loss: 1.1872 | Val F1: 0.1441 | Val Acc: 0.4048
Epoch 3/6 | Train Loss: 1.1771 | Val Loss: 1.1573 | Val F1: 0.1471 | Val Acc: 0.4167
Epoch 4/6 | Train Loss: 1.1465 | Val Loss: 1.1958 | Val F1: 0.1441 | Val Acc: 0.4048
Epoch 5/6 | Train Loss: 1.1607 | Val Loss: 1.1371 | Val F1: 0.1441 | Val Acc: 0.4048
Epoch 6/6 | Train Loss: 1.1600 | Val Loss: 1.1386 | Val F1: 0.1471 | Val Acc: 0.4167


In [117]:
test_df.iloc[17]

docid                                 nyt_20130322_strange_computer
verb1                                                        expect
verb2                                                           say
eiid1                                                           E19
eiid2                                                           E14
relation                                                      AFTER
context_window    [TIMEX3]Now[/TIMEX3], Lockheed Martin which bo...
relation_id                                                       2
Name: 920, dtype: object

In [118]:
context = test_df.iloc[17]["context_window"]

model.eval()
enc = tokenizer(
    context,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)

with torch.no_grad():
    logits = model(input_ids=input_ids, attention_mask=attention_mask)
    print(logits)
    pred_id = torch.argmax(logits, dim=1).item()

pred_label = id2relation[pred_id]
print("Predicted label:", pred_label)

tensor([[-0.0870,  1.1583,  1.1172, -2.1464]], device='cuda:0')
Predicted label: BEFORE


In [34]:
num_labels

4

In [37]:
relation2id

{'VAGUE': 0, 'BEFORE': 1, 'AFTER': 2, 'EQUAL': 3}